# 🎯 Strategy Development and Optimization
## Quantitative Trading Strategy MVP Project

This notebook develops and optimizes trading strategies using the processed data from our exploration.

### Objectives:
1. **Momentum Strategy** - Implement and optimize dual MA crossover with RSI
2. **Mean Reversion Strategy** - Develop Bollinger Bands and statistical arbitrage
3. **Risk Models** - Apply GARCH and factor models
4. **Parameter Optimization** - Find optimal strategy parameters
5. **Multi-Asset Strategies** - Portfolio-level strategy implementation


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
from itertools import product
import pickle
from pathlib import Path
import sys
import asyncio

warnings.filterwarnings('ignore')

# Windows: ensure selector event loop policy for ZMQ compatibility
if sys.platform.startswith('win'):
    try:
        from asyncio import WindowsSelectorEventLoopPolicy
        asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())
    except Exception:
        pass

# Ensure project src directory (parent of this notebook) is on sys.path
project_src = str(Path().resolve().parent)
if project_src not in sys.path:
    sys.path.append(project_src)

from models.momentum_strategy import MomentumStrategy
from models.mean_reversion import MeanReversionStrategy
from models.risk_models import RiskModels
from data_collection.yfinance_collector import YFinanceCollector
from data_collection.data_processor import DataProcessor

print("🎯 Setup complete! Ready for strategy development.")


## 1. Load Data and Previous Analysis


In [ ]:
# Load processed data from exploration notebook
try:
    with open('../data/exploration_summary.pkl', 'rb') as f:
        summary_data = pickle.load(f)

    returns_matrix = summary_data['returns_matrix']
    correlation_matrix = summary_data['correlation_matrix']
    risk_return_metrics = summary_data['risk_return_metrics']
    focus_data = summary_data['focus_ticker_processed']

    print("✅ Loaded processed data from exploration phase")
    print(f"   Returns matrix: {returns_matrix.shape}")
    print(f"   Focus ticker data: {focus_data.shape}")

except FileNotFoundError:
    print("⚠️ Exploration data not found. Running fresh data collection...")

    # Fresh data collection
    collector = YFinanceCollector()
    processor = DataProcessor()

    # Download AAPL data for focus analysis
    import yfinance as yf
    focus_data = yf.download('AAPL', start='2020-01-01', end='2024-01-01')
    focus_data = processor.calculate_technical_indicators(focus_data)

    print("✅ Fresh data loaded and processed")

FOCUS_TICKER = 'AAPL'
print(f"\n📊 Focus ticker for strategy development: {FOCUS_TICKER}")


## 2. Momentum Strategy Development


In [ ]:
# Initialize basic momentum strategy
momentum_strategy = MomentumStrategy(
    short_window=20,
    long_window=50,
    rsi_period=14,
    max_position_size=0.1
)

# Run basic backtest
signals_df, performance = momentum_strategy.backtest(focus_data)

print("📈 Basic Momentum Strategy Results:")
print(f"   Total Return: {performance['total_return']:.2%}")
print(f"   Sharpe Ratio: {performance['sharpe_ratio']:.3f}")
print(f"   Max Drawdown: {performance['max_drawdown']:.2%}")
print(f"   Win Rate: {performance['win_rate']:.1%}")
print(f"   Total Trades: {performance['total_trades']}")

# Visualize strategy performance
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=('Price & Signals', 'Strategy vs Benchmark', 'Position Sizes'),
    vertical_spacing=0.08,
    row_heights=[0.4, 0.3, 0.3]
)

# Price and signals
fig.add_trace(
    go.Scatter(
        x=signals_df.index,
        y=signals_df['Close'],
        name='Price',
        line=dict(color='black')
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=signals_df.index,
        y=signals_df['MA_Short'],
        name='MA Short',
        line=dict(color='blue')
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=signals_df.index,
        y=signals_df['MA_Long'],
        name='MA Long',
        line=dict(color='red')
    ),
    row=1, col=1
)

# Add buy/sell signals
buy_signals = signals_df[signals_df['Signal'] == 1]
sell_signals = signals_df[signals_df['Signal'] == -1]

if not buy_signals.empty:
    fig.add_trace(
        go.Scatter(
            x=buy_signals.index,
            y=buy_signals['Close'],
            mode='markers',
            marker=dict(color='green', size=8, symbol='triangle-up'),
            name='Buy Signal'
        ),
        row=1, col=1
    )

if not sell_signals.empty:
    fig.add_trace(
        go.Scatter(
            x=sell_signals.index,
            y=sell_signals['Close'],
            mode='markers',
            marker=dict(color='red', size=8, symbol='triangle-down'),
            name='Sell Signal'
        ),
        row=1, col=1
    )

# Strategy vs benchmark performance
fig.add_trace(
    go.Scatter(
        x=signals_df.index,
        y=signals_df['Cumulative_Returns'],
        name='Strategy',
        line=dict(color='green', width=2)
    ),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(
        x=signals_df.index,
        y=signals_df['Benchmark_Returns'],
        name='Buy & Hold',
        line=dict(color='gray', width=2)
    ),
    row=2, col=1
)

# Position sizes
fig.add_trace(
    go.Scatter(
        x=signals_df.index,
        y=signals_df['Position_Size'],
        name='Position Size',
        line=dict(color='purple'),
        fill='tonexty'
    ),
    row=3, col=1
)

fig.update_layout(
    title=f'{FOCUS_TICKER} Momentum Strategy Performance',
    height=800,
    showlegend=True
)

fig.show()
